## Paralelización de simulaciones

En muchos problemas científicos necesitamos ejecutar el mismo modelo muchas veces con distintos parámetros.

Ejemplo:

- explorar valores de beta  
- probar distintos escenarios de movilidad  

---

Estas simulaciones son independientes:

> podemos ejecutarlas en paralelo

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
from multiprocessing import Pool, cpu_count
import time

from EpiCommute import SIRModel

matplotlib.rc('figure', dpi=200)
np.random.seed = 1234

In [ ]:
def run_model(
        mobility,
        subpopulation_sizes,
        beta=0.375, 
        mu=1.0/8.0, 
        outbreak_source=0,
        initial_infected = 10,
        T_max = 150,
        VERBOSE=False
        ):
    
    """
    Ejecuta una simulación con EpiCommute y devuelve una métrica.
    """

    M = len(subpopulation_sizes)

    # Basic reproduction number
    R0   = beta / mu 

    # Initialize the model
    model = SIRModel(
                mobility,
                subpopulation_sizes,
                R0=R0,
                mu=mu,
                outbreak_source=outbreak_source, # random outbreak location
                dt=0.1,                          # simulation time interval
                dt_save=1,                       # time interval when to save observables
                I0=initial_infected,             # number of initial infected
                T_max=T_max,                     # Max simulation time
                VERBOSE=VERBOSE                  # print verbose output
            )

    if VERBOSE:
        print(f"- Running Epicommute using beta={beta} mu={mu} outbreak source={outbreak_source} initial infected={initial_infected}")
    
    result = model.run_simulation()

    
    region_ids = [f"R{i}" for i in range(M)]

    time = result['t']

    # Variables (time x region)
    I = np.array(result['I'])  # shape: (T, M)
    S = np.array(result['S'])  # shape: (T, M)
    R = np.array(result['R'])  # shape: (T, M)

    # Create dataset
    ds = xr.Dataset(
        {
            "I": (["time", "region"], I),
            "S": (["time", "region"], S),
            "R": (["time", "region"], R),
        },
        coords={
            "time": time,
            "region": region_ids,
        }
    )

    # Optional: add metadata
    ds.attrs["description"] = "Metapopulation SIR simulation"
    ds.attrs["model"] = "SIRModel (EpiCommute)"

    return ds

In [ ]:
M = 20    # Number of subpopulations

# Initialize a random mobility matrix
mobility = np.random.rand(M, M)

# Choose random subpopulation sizes
subpopulation_sizes = np.random.randint(20, 100, M)

## Ejecución secuencial

In [ ]:
betas = np.linspace(0.2, 0.5, 3)

start = time.time()

result_list = []
for beta in betas:
    ds = run_model(mobility, subpopulation_sizes, beta=beta, VERBOSE=True)
    peak = ds["I"].sum(dim="region").max().item()
    result_list.append((beta,peak))
    print("--------------------")


df_seq = pd.DataFrame(result_list, columns=['beta', 'peak'])
end = time.time()

print(f"Tiempo secuencial: {end - start:.2f} s")

df_seq

## Ejecución en paralelo

In [ ]:
def run_model_wrapper(beta):
    ds = run_model(
        mobility=mobility,
        subpopulation_sizes=subpopulation_sizes,
        beta=beta
    )

    # extraemos métrica
    peak = ds["I"].sum(dim="region").max().item()

    return {
        "beta": beta,
        "peak": peak
    }

In [ ]:
from multiprocessing import Pool
import numpy as np
import time

CPUS = 10

betas = np.linspace(0.2, 0.5, 20)

start = time.time()
results = []
with Pool(processes=CPUS) as pool:
    results = pool.map(run_model_wrapper, betas)

end = time.time()

print(f"Tiempo paralelo: {end - start:.2f} s")

df_par = pd.DataFrame(results)
df_par.head()

In [ ]:
import matplotlib.pyplot as plt

plt.plot(df_par["beta"], df_par["peak"], marker="o")
plt.xlabel("beta")
plt.ylabel("Peak infected")
plt.title("Exploración de parámetros")
plt.show()

# Versión con múltiples parámetros

In [ ]:
def run_model_params(params):
    beta, outbreak_source = params

    ds = run_model(
        mobility=mobility,
        subpopulation_sizes=subpopulation_sizes,
        beta=beta,
        outbreak_source=outbreak_source
    )

    peak = ds["I"].sum(dim="region").max().item()

    return {
        "beta": beta,
        "source": outbreak_source,
        "peak": peak
    }

In [ ]:
betas = np.linspace(0.2, 0.5, 6)
sources = [0, 5, 10]

param_grid = [(b, s) for b in betas for s in sources]

In [ ]:
with Pool(processes=CPUS) as pool:
    results = pool.map(run_model_params, param_grid)

df = pd.DataFrame(results)

In [ ]:
for s in df["source"].unique():
    subset = df[df["source"] == s]
    plt.plot(subset["beta"], subset["peak"], label=f"source={s}")

plt.legend()
plt.xlabel("beta")
plt.ylabel("peak")
plt.title("Exploración de parámetros")
plt.show()